# 2. Conformal prediction models and SHAP

Trains the models on the splits from notebook 1:
- **Swedish model**: trained and calibrated on SMSReg
- **Global model**: trained and calibrated on all MSBase training countries pooled
- **Country-specific model**: trained and calibrated within one country
- **Hybrid model**: trained on the other countries, calibrated on the target country

Each is a random forest (100 trees, `balanced_subsample`, `min_samples_leaf=5`) inside a Mondrian ICP from nonconformist. The Swedish model uses inverse probability as nonconformity measure, the MSBase models use margin (same as the published models). Calibration data is removed before saving.

After training it computes, for every test country: p-values for all models, the assigned model, SHAP values and the Lorscheider comparison. Output goes to `custom_run/` in the same layout as `files/`, so notebook 3 can read it with `MODEL_OUTPUTS_DIR = "custom_run/model_outputs"`.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os

import joblib
import numpy as np
import pandas as pd
import shap
import tqdm
from nonconformist.base import ClassifierAdapter
from nonconformist.icp import IcpClassifier
from nonconformist.nc import ClassifierNc, InverseProbabilityErrFunc, MarginErrFunc
from sklearn.ensemble import RandomForestClassifier

from harmonise_utils import (FEATURE_NAMES, FEATURES, LORSCHEIDER_COLUMNS, REFERENCE_COUNTRY, assign_models,
                             benchmark_country, jsd_matrix, lambda_function, load_model, model_file, shap_tag)

import warnings
warnings.filterwarnings("ignore")

/Users/akspa742/Desktop/delete/git_fix/HARMONISE/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_DIR = "custom_run"
DATA_DIR = f"{RUN_DIR}/data"
MODEL_DIR = f"{RUN_DIR}/models"
SHAP_DIR = f"{RUN_DIR}/shap"
OUT_DIR = f"{RUN_DIR}/model_outputs"
for d in [MODEL_DIR, SHAP_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 0
SHAP_SAMPLES = 2500

In [3]:
def read(registry, split):
    return pd.read_csv(f"{DATA_DIR}/{registry}/{split}.csv", parse_dates=["visit_date"])


ms_train, ms_calib, ms_test = (read("msbase", s) for s in ["train", "calib", "test"])
se_train, se_calib, se_test = (read("smsreg", s) for s in ["train", "calib", "test"])

suff = pd.read_csv(f"{DATA_DIR}/data_sufficiency.csv")
trainable = sorted(suff.query("registry == 'msbase' and trainable")["country"])
assert len(trainable) >= 2, "need at least two MSBase training countries for the hybrid models"

test_sets = dict(tuple(ms_test.groupby("country")))
test_sets[REFERENCE_COUNTRY] = se_test
relapses = pd.concat([pd.read_csv(f"{DATA_DIR}/{r}/relapses.csv", parse_dates=["relapse_date"]) for r in ["msbase", "smsreg"]])

print("training countries:", trainable)
print("test countries:", sorted(test_sets))

training countries: ['AU', 'CA', 'IT']
test countries: ['AR', 'AU', 'CA', 'IT', 'JP', 'SE']


## Train

In [4]:
def strip(icp):
    # remove calibration data and cached predictions so the saved model holds no patient data
    icp.cal_x, icp.cal_y = None, None
    for part in (icp.nc_function, icp.nc_function.model):
        for attr in ("last_x", "last_y", "last_prediction"):
            if hasattr(part, attr):
                setattr(part, attr, None)
        part.clean = False
    return icp


def train_icp(train, calib, err_func, name):
    rf = RandomForestClassifier(n_estimators=100, random_state=0, class_weight="balanced_subsample",
                                criterion="gini", min_samples_leaf=5)
    icp = IcpClassifier(ClassifierNc(ClassifierAdapter(rf), err_func), condition=lambda_function)
    icp.fit(train[FEATURES].values, train["y_label"].values)
    icp.calibrate(calib[FEATURES].values, calib["y_label"].values)
    joblib.dump(strip(icp), f"{MODEL_DIR}/{name}.joblib")
    return icp


def countries(df, ccs):
    return df[df["country"].isin(ccs)]


swedish = train_icp(se_train, se_calib, InverseProbabilityErrFunc(), "swedish")
global_model = train_icp(countries(ms_train, trainable), countries(ms_calib, trainable), MarginErrFunc(), "global")

individual, hybrid = {}, {}
for cc in tqdm.tqdm(trainable):
    others = [c for c in trainable if c != cc]
    individual[cc] = train_icp(countries(ms_train, [cc]), countries(ms_calib, [cc]), MarginErrFunc(), f"country_specific_{cc}")
    hybrid[cc] = train_icp(countries(ms_train, others), countries(ms_calib, [cc]), MarginErrFunc(), f"hybrid_{cc}")

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:00<00:00,  4.59it/s]

 67%|██████▋   | 2/3 [00:00<00:00,  4.58it/s]

100%|██████████| 3/3 [00:00<00:00,  4.54it/s]

100%|██████████| 3/3 [00:00<00:00,  4.55it/s]

## p-values on the test sets

All models are applied to all test countries, except the country-specific models which only predict their own country.

In [5]:
def predict(icp, df):
    np.random.seed(SEED)  # the p-values are smoothed with random noise
    p = icp.predict(df[FEATURES].values)
    strip(icp)
    return {"p_values": p.tolist(), "y_label": df["y_label"].astype(int).tolist()}


results = {
    "swedish": {"se": {cc: predict(swedish, df) for cc, df in test_sets.items()}},
    "global": {"global": {cc: predict(global_model, df) for cc, df in test_sets.items()}},
    "country_specific": {cc: {cc: predict(m, test_sets[cc])} for cc, m in individual.items()},
    "hybrid": {cc: {t: predict(m, df) for t, df in test_sets.items()} for cc, m in hybrid.items()},
}
for name, res in results.items():
    with open(f"{OUT_DIR}/{name}_results.json", "w") as f:
        json.dump(res, f)

## Model assignment

In [6]:
def entry_for(model, cc):
    if model.startswith("Hybrid"):
        return results["hybrid"][model.split()[1]][cc]
    if model == "Global model":
        return results["global"]["global"][cc]
    return results["swedish"]["se"][cc]


assignment = assign_models(jsd_matrix(entry_for, sorted(test_sets), trainable), trainable)
with open(f"{OUT_DIR}/model_assignment.json", "w") as f:
    json.dump(assignment, f, indent=1)
assignment

{'AR': 'Global model',
 'AU': 'Hybrid AU',
 'CA': 'Hybrid CA',
 'IT': 'Hybrid IT',
 'JP': 'Global model',
 'SE': 'Country-specific SE'}

## SHAP and Lorscheider comparison

Both use the assigned model of each country. SHAP is computed on up to 2500 test visits (SPMS class).

In [7]:
counts, diffs = {}, {}
for cc, df in tqdm.tqdm(sorted(test_sets.items())):
    icp = load_model(model_file(MODEL_DIR, assignment[cc]))

    sample = df.sample(min(SHAP_SAMPLES, len(df)), random_state=SEED)
    explainer = shap.TreeExplainer(icp.nc_function.model.model, feature_perturbation="tree_path_dependent")
    sv = explainer(sample[FEATURES].values)[..., 1]
    sv.feature_names = FEATURE_NAMES
    joblib.dump(sv, f"{SHAP_DIR}/data_{cc}_model_{shap_tag(assignment[cc])}.joblib")

    res = benchmark_country(df, relapses, icp)
    counts[cc] = res["counts"]
    diffs[cc] = {"lor_diff": res["lor_diff"], "pred_diff": res["pred_diff"]}

pd.DataFrame(counts).T[LORSCHEIDER_COLUMNS].to_csv(f"{OUT_DIR}/model_lor_cm.csv")
with open(f"{OUT_DIR}/lor_pred_visit_diff_results.json", "w") as f:
    json.dump(diffs, f)

  0%|          | 0/6 [00:00<?, ?it/s]

 17%|█▋        | 1/6 [00:00<00:01,  4.78it/s]

 33%|███▎      | 2/6 [00:00<00:00,  5.38it/s]

 50%|█████     | 3/6 [00:00<00:00,  5.85it/s]

 67%|██████▋   | 4/6 [00:00<00:00,  6.06it/s]

 83%|████████▎ | 5/6 [00:00<00:00,  5.33it/s]

100%|██████████| 6/6 [00:01<00:00,  5.86it/s]

100%|██████████| 6/6 [00:01<00:00,  5.69it/s]